## **CLEANING PHASE**

Step 1 - Load the data and do an inspection first

In [1]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd

df = pd.read_csv("../data/raw_data.csv")

print(df.shape)

#shows you column dtypes, non-null counts per column
df.info()
df.head(3)

#clean per-column count of missing values
df.isnull().sum()

(418, 8)
<class 'pandas.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   title        418 non-null    str  
 1   company      418 non-null    str  
 2   location     366 non-null    str  
 3   tags         418 non-null    str  
 4   description  418 non-null    str  
 5   apply_url    418 non-null    str  
 6   date_posted  418 non-null    str  
 7   search_tag   418 non-null    str  
dtypes: str(8)
memory usage: 26.3 KB


title           0
company         0
location       52
tags            0
description     0
apply_url       0
date_posted     0
search_tag      0
dtype: int64

Step 2 - Handle Missing values

In [3]:
df["company"] = df["company"].fillna("Not specified")
df["location"] = df["location"].fillna("Not specified")
df["tags"] = df["tags"].fillna("")
df["description"] = df["description"].fillna("")

Step 3 - Drop duplicate postings

In [4]:
before = len(df)
df = df.drop_duplicates(subset=["title", "company", "apply_url"]) #The apply URL is unique per posting
after = len(df)
print(f"Removed {before - after} duplicate rows")

Removed 0 duplicate rows


Step 4 - Lowercase and strip text columns

In [5]:
text_cols = ["title", "company", "location", "tags", "description"]

#.astype(str) guarantees that every value is text
for col in text_cols:
    df[col] = df[col].astype(str).str.lower().str.strip()

Step 5 - Remove leftover HTML/whitespace artifacts

In [6]:
import re

def clean_text_artifacts(text):
    text = re.sub(r"\s+", " ", text)      # collapse multiple spaces/newlines into one
    text = text.replace("\xa0", " ")       # remove non-breaking space characters
    text = text.strip()
    return text

for col in text_cols:
    df[col] = df[col].apply(clean_text_artifacts)

Step 6 - Check the result

In [26]:
df[["title", "company", "location", "description"]].sample(5)

,title,company,location,description
254,social media and influencer manager,"pete &amp; gerry's organics, llc","chicago, chicago, illinois, united states","job type full-time description healthy hens, h..."
127,team leader housekeeping,hyatt regency,"dehradun,",organization- hyatt regency dehradun summary y...
233,associate account strategist donut studios,new engen,"new york, new york, new york, united states","why donut studios? at new engen, we help brand..."
369,product marketing intern,everlaw,oakland,"as a product marketing intern at everlaw, you ..."
201,senior data engineer,lalamove,kuala lumpur,"at lalamove, we believe in the power of commun..."


In [9]:
print((df["description"].str.len() < 20).sum(), "postings have very short descriptions")

0 postings have very short descriptions


Step 7 - Save the cleaned version

In [10]:
df.to_csv("../data/cleaned_data.csv", index=False)
print(f"Final cleaned dataset: {len(df)} rows")

Final cleaned dataset: 418 rows


## **SKILLS EXTRACTION**

Step 1 - Load you cleaned data

In [5]:
import pandas as pd
import re

df = pd.read_csv("../data/cleaned_data.csv")
df.shape

(418, 8)

Step 2 - Define you skill list, grouped by category

In [6]:
technical_skills = [
    "python", "sql", "r", "java", "javascript", "c++", "c#", "scala",
    "excel", "power bi", "tableau", "looker",
    "aws", "azure", "gcp", "docker", "kubernetes",
    "machine learning", "deep learning", "nlp", "statistics",
    "spark", "hadoop", "airflow", "git", "linux",
    "figma", "photoshop", "seo", "html", "css"
]

soft_skills = [
    "communication", "collaboration", "teamwork", "leadership",
    "problem solving", "stakeholder management", "critical thinking",
    "time management", "adaptability", "presentation"
]

Step 3 - Word-boundary matching

In [7]:
def skill_present(description, skill):
    """
    Checks if a skill appears as a standalone word/phrase,
    using lookaround instead of \\b so it works correctly even
    for skills ending in symbols like 'c++' or 'c#'.
    """
    pattern = r"(?<!\w)" + re.escape(skill) + r"(?!\w)"
    return bool(re.search(pattern, description))

In [8]:
sample = df["description"].iloc[0] 
print(skill_present(sample, "python"))
print(skill_present(sample, "r"))

False
False


Step 4 - Build one boolean column per skill

In [9]:
for skill in technical_skills:
    df[skill] = df["description"].apply(lambda desc: skill_present(desc, skill))

for skill in soft_skills:
    df[skill] = df["description"].apply(lambda desc: skill_present(desc, skill))

Step 5 - Small check

In [13]:
df[["title"] + technical_skills[:5]].sample(5)

,title,python,sql,r,java,javascript
49,crm sr analyst,True,True,False,False,False
223,technician 3,False,False,False,False,False
312,user researcher,False,False,False,False,False
203,senior devops architect,False,True,False,False,False
355,product manager retail lending &amp; enablement,False,False,False,False,False


Step 6 - Count skill frequency across all postings

In [16]:
#Technical Skills Count
skill_counts = df[technical_skills].sum().sort_values(ascending=False)
print(skill_counts)

sql                 50
excel               48
python              46
aws                 17
power bi            15
figma               15
azure               13
machine learning    13
git                 13
tableau             12
r                   12
statistics          10
gcp                 10
linux                9
looker               8
javascript           8
html                 8
spark                7
java                 7
css                  7
seo                  7
kubernetes           7
docker               7
c#                   4
photoshop            4
scala                3
c++                  3
airflow              3
nlp                  1
hadoop               0
deep learning        0
dtype: int64


In [17]:
#Soft Skills Count
soft_counts = df[soft_skills].sum().sort_values(ascending=False)
print(soft_counts)

communication             175
leadership                 91
collaboration              74
time management            21
presentation               21
teamwork                   19
adaptability               15
stakeholder management     12
problem solving            10
critical thinking           4
dtype: int64


Step 7 - Convert counts to percentages

In [18]:
#Technical skills percentages
total_postings = len(df)
skill_percentages = (skill_counts / total_postings * 100).round(1)
print(skill_percentages)

sql                 12.0
excel               11.5
python              11.0
aws                  4.1
power bi             3.6
figma                3.6
azure                3.1
machine learning     3.1
git                  3.1
tableau              2.9
r                    2.9
statistics           2.4
gcp                  2.4
linux                2.2
looker               1.9
javascript           1.9
html                 1.9
spark                1.7
java                 1.7
css                  1.7
seo                  1.7
kubernetes           1.7
docker               1.7
c#                   1.0
photoshop            1.0
scala                0.7
c++                  0.7
airflow              0.7
nlp                  0.2
hadoop               0.0
deep learning        0.0
dtype: float64


In [19]:
#Soft skills percentages
soft_percentages = (soft_counts / total_postings * 100).round(1)
print(soft_percentages)

communication             41.9
leadership                21.8
collaboration             17.7
time management            5.0
presentation               5.0
teamwork                   4.5
adaptability               3.6
stakeholder management     2.9
problem solving            2.4
critical thinking          1.0
dtype: float64


In [20]:
combined_percentages = pd.concat([skill_percentages, soft_percentages]).sort_values(ascending=False)
print(combined_percentages)

communication             41.9
leadership                21.8
collaboration             17.7
sql                       12.0
excel                     11.5
python                    11.0
time management            5.0
presentation               5.0
teamwork                   4.5
aws                        4.1
power bi                   3.6
adaptability               3.6
figma                      3.6
git                        3.1
machine learning           3.1
azure                      3.1
tableau                    2.9
stakeholder management     2.9
r                          2.9
statistics                 2.4
gcp                        2.4
problem solving            2.4
linux                      2.2
javascript                 1.9
looker                     1.9
html                       1.9
css                        1.7
docker                     1.7
kubernetes                 1.7
seo                        1.7
spark                      1.7
java                       1.7
c#      

Step 8 - Save the skill-augmented dataset

In [21]:
df.to_csv("../data/skills_data.csv", index=False)
print("Saved skills_data.csv with", len(df), "rows and", len(df.columns), "columns")

combined_percentages.to_csv("../data/skill_percentages.csv", header=["percentage"])
print("Saved skill_percentages.csv")

Saved skills_data.csv with 418 rows and 49 columns
Saved skill_percentages.csv
